# Learning what normal looks like

MichAl Academy, lesson 3.5.

Run each cell with **Shift+Enter**.

Lesson 2.14 found anomalies by asking which points sit apart from the others.
This notebook asks a different question: train a model to rebuild its input
through a narrow gap, then score anything by how badly it comes back.

No labels are used to build the detector.


In [ ]:
import numpy as np
import torch
from sklearn.datasets import load_digits
from sklearn.decomposition import PCA
from sklearn.ensemble import IsolationForest
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.model_selection import train_test_split

torch.set_num_threads(1)
rng = np.random.default_rng(0)

X, y = load_digits(return_X_y=True)
X = X / 16.0
print(f"digits: {X.shape}")


In [ ]:
def train_ae(Xn, bottleneck, seed=0, epochs=150, lr=1e-3):
    """64 -> 32 -> bottleneck -> 32 -> 64, trained to reproduce its input."""
    torch.manual_seed(seed)
    ae = torch.nn.Sequential(
        torch.nn.Linear(64, 32), torch.nn.ReLU(),
        torch.nn.Linear(32, bottleneck), torch.nn.ReLU(),
        torch.nn.Linear(bottleneck, 32), torch.nn.ReLU(),
        torch.nn.Linear(32, 64))
    opt = torch.optim.Adam(ae.parameters(), lr=lr)
    Xt = torch.tensor(Xn, dtype=torch.float32)
    g = torch.Generator().manual_seed(seed)
    for _ in range(epochs):
        perm = torch.randperm(len(Xt), generator=g)
        for i in range(0, len(Xt), 64):
            idx = perm[i:i + 64]
            opt.zero_grad()
            ((ae(Xt[idx]) - Xt[idx]) ** 2).mean().backward()
            opt.step()
    return ae


def ae_error(ae, Xq):
    with torch.no_grad():
        t = torch.tensor(Xq, dtype=torch.float32)
        return ((ae(t) - t) ** 2).mean(dim=1).numpy()


def pca_error(pca, Xq):
    return ((pca.inverse_transform(pca.transform(Xq)) - Xq) ** 2).mean(axis=1)


def split_for(held_out):
    """Everything except one digit is normal. That digit is the anomaly."""
    normal, anom = X[y != held_out], X[y == held_out]
    n_tr, n_te = train_test_split(normal, test_size=0.3, random_state=0)
    Xq = np.vstack([n_te, anom])
    lab = np.concatenate([np.zeros(len(n_te)), np.ones(len(anom))])
    return n_tr, Xq, lab


## 1. One held-out digit, three detectors

The autoencoder never sees a 3. Neither does PCA, which does the same job with a
linear bottleneck and no training loop, nor Isolation Forest, which was the tool
in lesson 2.14.


In [ ]:
BOTTLENECK = 8
rows = []

print(f"{'held out':<10}{'AE auc':>9}{'PCA auc':>9}{'iF auc':>9}"
      f"{'AE ap':>9}{'PCA ap':>9}{'iF ap':>9}")
for d in range(10):
    n_tr, Xq, lab = split_for(d)

    s_ae = ae_error(train_ae(n_tr, BOTTLENECK), Xq)
    s_pca = pca_error(PCA(n_components=BOTTLENECK, random_state=0).fit(n_tr), Xq)
    s_if = -IsolationForest(random_state=0).fit(n_tr).score_samples(Xq)

    row = [roc_auc_score(lab, s) for s in (s_ae, s_pca, s_if)] + \
          [average_precision_score(lab, s) for s in (s_ae, s_pca, s_if)]
    rows.append(row)
    print(f"{d:<10}" + "".join(f"{v:>9.4f}" for v in row))

means = np.array(rows).mean(axis=0)
print(f"\n{'mean':<10}" + "".join(f"{v:>9.4f}" for v in means))


In [ ]:
wins = np.array(rows)[:, :3].argmax(axis=1)
for i, name in enumerate(["autoencoder", "PCA", "IsolationForest"]):
    print(f"{name:<16} best on {int((wins == i).sum())} of 10 held-out classes")


This is the first time in the course that the deep method takes it.

Lessons 2.6 and 2.16 both concluded that on a table, classical methods win, and
that still holds. What changed is the data. Sixty-four pixels with strong
spatial structure is exactly the case a linear projection cannot fully describe.

Isolation Forest comes last because it is looking for points in odd positions
and has no idea what a digit is. The autoencoder learned that.

## 2. The bottleneck is the detector


In [ ]:
print(f"{'bottleneck':<12}{'AE':>10}{'PCA':>10}")
for bn in (2, 4, 8, 16, 32):
    aes, pcas = [], []
    for d in range(10):
        n_tr, Xq, lab = split_for(d)
        aes.append(roc_auc_score(lab, ae_error(train_ae(n_tr, bn), Xq)))
        pcas.append(roc_auc_score(lab, pca_error(
            PCA(n_components=bn, random_state=0).fit(n_tr), Xq)))
    print(f"{bn:<12}{np.mean(aes):>10.4f}{np.mean(pcas):>10.4f}")


Both ends fail, for opposite reasons.

Squeeze too hard and the model cannot rebuild anything, so normal and unusual
both score badly and there is nothing to separate. Leave it too wide and the
model rebuilds anything at all, including a digit it has never seen, and again
there is nothing to separate.

Reconstruction quality is not the goal. The gap is.

## 3. Then the base rate arrives

Everything above scored a set that was about 27% anomalies. No security team has
ever seen that. Rescore the same models at rates you might actually meet.


In [ ]:
RATES = (0.268, 0.10, 0.02, 0.01)
auc_at = {r: [] for r in RATES}
ap_at = {r: [] for r in RATES}

for d in range(10):
    normal, anom = X[y != d], X[y == d]
    n_tr, n_te = train_test_split(normal, test_size=0.3, random_state=0)
    ae = train_ae(n_tr, BOTTLENECK)

    for rate in RATES:
        k = min(int(round(len(n_te) * rate / (1 - rate))), len(anom))
        idx = rng.choice(len(anom), size=k, replace=False)
        Xq = np.vstack([n_te, anom[idx]])
        lab = np.concatenate([np.zeros(len(n_te)), np.ones(k)])
        s = ae_error(ae, Xq)
        auc_at[rate].append(roc_auc_score(lab, s))
        ap_at[rate].append(average_precision_score(lab, s))

print(f"{'anomaly rate':<16}{'ROC-AUC':>10}{'average precision':>20}")
for rate in RATES:
    print(f"{rate * 100:>5.1f}%{'':<10}{np.mean(auc_at[rate]):>10.4f}"
          f"{np.mean(ap_at[rate]):>20.4f}")


The ROC-AUC barely moves. Average precision falls to a third of where it
started, on the same model producing the same scores.

Now the version that reaches an analyst.


In [ ]:
n_tr, _, _ = split_for(3)
normal, anom = X[y != 3], X[y == 3]
_, n_te = train_test_split(normal, test_size=0.3, random_state=0)

ae = train_ae(n_tr, BOTTLENECK)
k = int(round(len(n_te) * 0.01 / 0.99))
idx = rng.choice(len(anom), size=k, replace=False)
Xq = np.vstack([n_te, anom[idx]])
lab = np.concatenate([np.zeros(len(n_te)), np.ones(k)])

order = np.argsort(-ae_error(ae, Xq))
print(f"scored set: {len(Xq)} items, of which {k} are real anomalies\n")
for top in (5, 10, 20, 50):
    print(f"  of the {top:>2} worst reconstructions, "
          f"{int(lab[order[:top]].sum())} of {k} are real anomalies")


## 4. What you have

- An autoencoder learns normal without labels, and reconstruction error is the
  score.
- The bottleneck is not an implementation detail. Too narrow and nothing is
  rebuilt, too wide and everything is, and both destroy the detector.
- On pixels the deep method genuinely wins, which is a real change from Track 2
  and is about the data rather than about fashion.
- A model can be genuinely better than the alternative and still hand an analyst
  a queue with nothing in the top twenty. Both facts are true, and the second is
  an argument for a triage stage rather than for a different model.

Lesson 3.6 turns to data where the order carries the meaning.
